In [1]:
import sys
!{sys.executable} -m pip install google-api-python-client


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ── 셋업 셀: 커널 재시작할 때마다 이것부터 실행 ──
import json
from googleapiclient.discovery import build
from config import API_KEY   # config.py가 노트북과 같은 폴더에 있어야 함

yt = build("youtube", "v3", developerKey=API_KEY)
print("yt 클라이언트 준비 완료")

yt 클라이언트 준비 완료


In [5]:
import json

with open("raw_jeju_matjip_test.json", encoding="utf-8") as f:
    videos = json.load(f)

valid = [v for v in videos if len(v["description"]) > 100]
buckets = {"10만 이상": 0, "1만~10만": 0, "1천~1만": 0, "1천 미만": 0}
for v in valid:
    vc = v["view_count"]
    if vc >= 100_000: buckets["10만 이상"] += 1
    elif vc >= 10_000: buckets["1만~10만"] += 1
    elif vc >= 1_000: buckets["1천~1만"] += 1
    else: buckets["1천 미만"] += 1

print(f"유효 영상 {len(valid)}건의 조회수 분포:")
for k, n in buckets.items():
    print(f"  {k}: {n}건")

# 10만 컷의 대가: 뭘 잃는지 직접 보기
print("\n=== 10만 컷이 버리게 되는 영상들 ===")
for v in sorted(valid, key=lambda x: -x["view_count"]):
    if v["view_count"] < 100_000:
        print(f"  {v['view_count']:>8,} | {v['channel_title']} | {v['title'][:35]}")

유효 영상 20건의 조회수 분포:
  10만 이상: 15건
  1만~10만: 5건
  1천~1만: 0건
  1천 미만: 0건

=== 10만 컷이 버리게 되는 영상들 ===
    97,952 | 살란다 | 제주도 도민추천 제주 맛집 BEST ㅣ가심비 가성비 최고 실패없
    30,123 | 맛칼럼쭌선생 | 현지인도 극찬한 제주 맛집 3곳!
    23,153 | MPLAY : 엠플레이 | [📁제주도 맛집_찐 최종 모음] 고기부터 해산물까지 제주의 찐 
    18,131 | 취미부자 빈플렉스 | 제주도 현지인도 모르는 제주 맛집 딱 5끼 먹으러 2박3일 바이
    14,682 | 곰도리오 | 제주시 맛집, T맵 안내 순위로 본 맛집 1위~10위, 제주시 


In [9]:
# 노트북 셀 — "제주도 맛집" 10만+ 영상 카운트
# 쿼터: 페이지당 100유닛, 최대 10페이지 = ~1,000유닛

count, page_token, page = 0, None, 0
done = False

while not done and page < 10:  # API 한계 ~500건
    resp = yt.search().list(
        part="id", q="제주도 맛집", type="video",
        regionCode="KR", relevanceLanguage="ko",
        maxResults=50, order="viewCount",
        pageToken=page_token,
    ).execute()
    page += 1
    ids = [i["id"]["videoId"] for i in resp.get("items", [])]
    if not ids:
        break

    stats = yt.videos().list(part="statistics", id=",".join(ids)).execute()
    for v in stats.get("items", []):
        vc = int(v.get("statistics", {}).get("viewCount", 0))
        if vc >= 100_000:
            count += 1
        else:
            done = True  # 내림차순이라 여기부터는 전부 미만
            break

    page_token = resp.get("nextPageToken")
    if not page_token:
        break

print(f"'제주도 맛집' 조회수 10만 이상: {'500+ (API 한계 도달)' if not done else count}건")

'제주도 맛집' 조회수 10만 이상: 24건


In [10]:
# ── 키워드 탐색 셀: 크기 + 원료 품질 비교 ──
import re, json

KEYWORDS = [
    "제주 여행",
    "제주 명소",
    "제주 숨은 명소",
    "제주 사진 스팟",
    "제주 포토스팟",
    "제주 가볼만한곳",
    "제주 카페",
    "제주 스팟",
]

ADDR  = re.compile(r"(제주시|서귀포시|제주도)\s?\S*[읍면동리]|\S+로\s?\d+|\S+길\s?\d+")
INFO  = re.compile(r"영업시간|휴무|메뉴|주차|위치|입장료|\d{1,2}:\d{2}")
LISTY = re.compile(r"(첫\s?번째|두\s?번째|1\.|2\.|①|②|TOP\s?\d)", re.I)
SPAM  = re.compile(r"쿠팡|링크|광고|협찬|구매|inpock|smartstore")

all_raw, seen_ids, rows = [], set(), []

for kw in KEYWORDS:
    resp = yt.search().list(
        part="snippet", q=kw, type="video",
        regionCode="KR", relevanceLanguage="ko",
        maxResults=50, order="relevance",
        publishedAfter="2023-07-01T00:00:00Z",   # 폐업 리스크 컷
    ).execute()
    ids = [i["id"]["videoId"] for i in resp.get("items", [])]

    overlap = sum(1 for i in ids if i in seen_ids)   # 다른 키워드와 중복
    new_ids = [i for i in ids if i not in seen_ids]
    seen_ids.update(ids)

    videos = []
    for j in range(0, len(new_ids), 50):
        vr = yt.videos().list(part="snippet,statistics,contentDetails",
                              id=",".join(new_ids[j:j+50])).execute()
        for v in vr.get("items", []):
            sn = v["snippet"]
            videos.append({
                "video_id": v["id"], "title": sn["title"],
                "description": sn.get("description", ""),
                "channel_title": sn["channelTitle"],
                "view_count": int(v.get("statistics", {}).get("viewCount", 0)),
                "duration": v.get("contentDetails", {}).get("duration", ""),
                "keyword": kw,
            })
    all_raw.extend(videos)

    d100  = [v for v in videos if len(v["description"]) > 100]
    top   = sum(1 for v in d100 if ADDR.search(v["description"]) and INFO.search(v["description"]))
    listy = sum(1 for v in d100 if LISTY.search(v["description"]))
    spam  = sum(1 for v in d100 if SPAM.search(v["description"]))
    rows.append((kw, len(ids), overlap, len(d100), top, listy, spam))

print(f"{'키워드':<12} {'검색':>4} {'중복':>4} {'유효':>4} {'최상급':>5} {'리스트형':>6} {'스팸':>4}")
for r in rows:
    print(f"{r[0]:<12} {r[1]:>4} {r[2]:>4} {r[3]:>4} {r[4]:>5} {r[5]:>6} {r[6]:>4}")

with open("raw_explore.json", "w", encoding="utf-8") as f:
    json.dump(all_raw, f, ensure_ascii=False, indent=2)
print(f"\n신규 영상 총 {len(all_raw)}건 저장 → raw_explore.json")

키워드            검색   중복   유효   최상급   리스트형   스팸
제주 여행          50    0   47    15     14   16
제주 명소          50   13   21     9      6    6
제주 숨은 명소       50   15   18    10      3    4
제주 사진 스팟       50    9   18     3      2    5
제주 포토스팟        50   36    6     1      0    0
제주 가볼만한곳       50   34    6     2      2    1
제주 카페          50    2   31    12      4    7
제주 스팟          50   14   27     6      6    3

신규 영상 총 277건 저장 → raw_explore.json


In [ ]:
# pipeline/collect.py — 격자 전체 수집 (무손실 raw)
# 쿼터: 키워드 41개 × ~102유닛 ≈ 4,200유닛 (하루 한도의 42%)
import os, json, time
from datetime import datetime
from googleapiclient.discovery import build
from dotenv import load_dotenv
from config import API_KEY

yt = build("youtube", "v3", developerKey=API_KEY)

# ── 격자 4층 중 ①②③ (④ B트랙 채널은 별도 스크립트) ──
REGIONS = ["애월", "한림", "협재", "함덕", "월정리", "성산",
           "표선", "중문", "우도", "제주시", "서귀포"]
CATEGORIES = ["카페", "맛집", "포토존"]

KEYWORDS = (
    ["제주 여행", "제주 맛집", "제주 카페", "제주 숨은 명소", "제주 명소"]  # ② 광역
    + [f"{r} {c}" for r in REGIONS for c in CATEGORIES]                      # ① 본대 33개
    + ["제주 오름", "제주 해변", "제주 시장"]                                # ③ 테마
)

collected = {}   # video_id -> record  (무손실: 설명란 0자도 저장)
stats = []

for kw in KEYWORDS:
    try:
        resp = yt.search().list(
            part="snippet", q=kw, type="video",
            regionCode="KR", relevanceLanguage="ko",
            maxResults=50, order="relevance",
            publishedAfter="2023-07-01T00:00:00Z",
        ).execute()
        ids = [i["id"]["videoId"] for i in resp.get("items", [])]

        # 중복 = 버리지 않고 키워드 리스트에 추가 (다수결 신호 보존)
        new_ids = []
        for vid in ids:
            if vid in collected:
                collected[vid]["keywords"].append(kw)
            else:
                new_ids.append(vid)

        for j in range(0, len(new_ids), 50):
            vr = yt.videos().list(
                part="snippet,statistics,contentDetails",
                id=",".join(new_ids[j:j+50]),
            ).execute()
            for v in vr.get("items", []):
                sn = v["snippet"]
                collected[v["id"]] = {
                    "video_id": v["id"],
                    "title": sn["title"],
                    "description": sn.get("description", ""),
                    "tags": sn.get("tags", []),
                    "channel_id": sn["channelId"],
                    "channel_title": sn["channelTitle"],
                    "published_at": sn["publishedAt"],
                    "view_count": int(v.get("statistics", {}).get("viewCount", 0)),
                    "duration": v.get("contentDetails", {}).get("duration", ""),
                    "keywords": [kw],
                }
        stats.append((kw, len(ids), len(new_ids)))
        print(f"[{kw}] 검색 {len(ids)} / 신규 {len(new_ids)}")
        time.sleep(0.3)  # API 예의
    except Exception as e:
        stats.append((kw, -1, -1))
        print(f"[{kw}] 실패: {e} — 다음 키워드 계속")

# ── 저장 ──
os.makedirs("data/raw", exist_ok=True)
out = f"data/raw/raw_{datetime.now():%Y%m%d_%H%M}.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(list(collected.values()), f, ensure_ascii=False, indent=2)

# ── 요약 ──
total = len(collected)
multi = sum(1 for v in collected.values() if len(v["keywords"]) >= 2)
no_desc = sum(1 for v in collected.values() if len(v["description"]) <= 100)
print(f"\n총 {total}건 저장 → {out}")
print(f"복수 키워드 등장(다수결 후보): {multi}건")
print(f"설명란 100자 이하(B트랙 후보): {no_desc}건")
print(f"실패 키워드: {[s[0] for s in stats if s[1] == -1] or '없음'}")

[제주 여행] 검색 50 / 신규 50
[제주 맛집] 검색 50 / 신규 48
[제주 카페] 검색 50 / 신규 47
[제주 숨은 명소] 검색 50 / 신규 34
[제주 명소] 검색 50 / 신규 10
[애월 카페] 검색 50 / 신규 46
[애월 맛집] 검색 50 / 신규 39
[애월 포토존] 검색 50 / 신규 44
[한림 카페] 검색 50 / 신규 35
[한림 맛집] 검색 50 / 신규 32
[한림 포토존] 검색 50 / 신규 36
[협재 카페] 검색 50 / 신규 32
[협재 맛집] 검색 50 / 신규 26
[협재 포토존] 검색 50 / 신규 22
[함덕 카페] 검색 50 / 신규 42
[함덕 맛집] 검색 50 / 신규 46
[함덕 포토존] 검색 50 / 신규 22
[월정리 카페] 검색 50 / 신규 33
[월정리 맛집] 검색 50 / 신규 26
[월정리 포토존] 검색 37 / 신규 32
[성산 카페] 검색 50 / 신규 40
[성산 맛집] 검색 50 / 신규 33
[성산 포토존] 검색 50 / 신규 35
[표선 카페] 검색 50 / 신규 38
[표선 맛집] 검색 50 / 신규 26
[표선 포토존] 검색 50 / 신규 21
[중문 카페] 검색 50 / 신규 37
[중문 맛집] 검색 50 / 신규 26
[중문 포토존] 검색 50 / 신규 35
[우도 카페] 검색 50 / 신규 48
[우도 맛집] 검색 50 / 신규 34
[우도 포토존] 검색 50 / 신규 30
[제주시 카페] 검색 50 / 신규 11
[제주시 맛집] 검색 50 / 신규 17
[제주시 포토존] 검색 50 / 신규 28
[서귀포 카페] 검색 50 / 신규 23
[서귀포 맛집] 검색 50 / 신규 31
[서귀포 포토존] 검색 50 / 신규 17
[제주 오름] 검색 50 / 신규 44
[제주 해변] 검색 50 / 신규 37
[제주 시장] 검색 50 / 신규 46

총 1359건 저장 → data/raw/raw_20260707_1006.json
복수 키워드 등장(다수결 후보): 271건
설명란 1

: 

In [1]:
# 노트북 셀 — 지오코딩 경로 1 규모 실측
import json, re

with open("data/raw/raw_20260707_1006.json", encoding="utf-8") as f:
    raw = json.load(f)

ADDR = re.compile(r"제주[^\n]{0,20}(시|읍|면)[^\n]{0,30}(로|길)\s?\d+")

hits = []
for v in raw:
    m = ADDR.search(v["description"])
    if m:
        hits.append((v["channel_title"], m.group(0).strip()))

print(f"전체 {len(raw)}건 중 도로명 주소 포함: {len(hits)}건")
print("\n=== 추출된 주소 샘플 15개 ===")
for ch, addr in hits[:15]:
    print(f"  {ch} | {addr}")

전체 1359건 중 도로명 주소 포함: 218건

=== 추출된 주소 샘플 15개 ===
  여행작가 봄비 | 제주시 구좌읍 김녕로1길 51
  세밍웨이 saymingway | 제주 서귀포시 남원읍 남조로 52
  승G공주 | 제주 제주시 오현길 2
  푸글 | 제주 제주시 애월읍 가문동길 4
  살란다 | 제주시 구좌읍 세화8길 7
  제주먹킷 | 제주 서귀포시 안덕면 한창로 137
  냠플렉스 | 제주 서귀포시 토평로 24
  제주야미소영 | 제주시 가령로 4길 27
  술맛여행놈 | 제주 서귀포시 성산읍 성산중앙로 69
  제주먹킷 | 제주 서귀포시 태평로 152
  제주요 | 제주 제주시 서광로32길 20
  제주야미소영 | 제주 서귀포시 안덕면 화순중앙로 124번길 26
  Haruu | 제주 제주시 탑동로 17
  플레이 요니 Play Yoni | 제주 서귀포시 안덕면 화순중앙로124번길 26
  제주야미소영 | 제주시 조천읍 번영로 1218


In [2]:
# 노트북 셀 — 주소 기준 다수결 실측
import json, re
from collections import defaultdict

with open("data/raw/raw_20260707_1006.json", encoding="utf-8") as f:
    raw = json.load(f)

ADDR = re.compile(r"제주[^\n]{0,20}(?:시|읍|면)[^\n]{0,30}(?:로|길)\s?\d+(?:[-\s]?\d+)?(?:번길\s?\d+)?")

def normalize(a):
    # 표기 차이 흡수: 공백 제거, "제주특별자치도" 접두 제거
    return re.sub(r"\s+", "", a.replace("제주특별자치도", "제주"))

addr_videos = defaultdict(list)    # 정규화 주소 -> [(채널, video_id, 원문주소)]

for v in raw:
    # findall: 리스트형 영상은 주소가 여러 개
    for m in set(ADDR.findall(v["description"]) if False else
                 [x.group(0) for x in ADDR.finditer(v["description"])]):
        addr_videos[normalize(m)].append((v["channel_title"], v["video_id"], m.strip()))

multi = {a: lst for a, lst in addr_videos.items() if len(lst) >= 2}
multi_ch = {a: lst for a, lst in multi.items()
            if len(set(ch for ch, _, _ in lst)) >= 2}   # 서로 다른 채널

print(f"고유 주소: {len(addr_videos)}개")
print(f"2회 이상 등장: {len(multi)}개")
print(f"서로 다른 채널 2곳 이상: {len(multi_ch)}개  ← 진짜 다수결")

print("\n=== 다수결 상위 10 (채널 수 기준) ===")
ranked = sorted(multi_ch.items(),
                key=lambda x: len(set(ch for ch, _, _ in x[1])), reverse=True)
for addr, lst in ranked[:10]:
    chans = sorted(set(ch for ch, _, _ in lst))
    print(f"\n📍 {lst[0][2]}  — 영상 {len(lst)}개 / 채널 {len(chans)}곳")
    print(f"   {', '.join(chans[:5])}")

고유 주소: 533개
2회 이상 등장: 67개
서로 다른 채널 2곳 이상: 55개  ← 진짜 다수결

=== 다수결 상위 10 (채널 수 기준) ===

📍 제주 제주시 한림읍 한림서길 18  — 영상 4개 / 채널 4곳
   럭키커플트립 Lucky Couple Trip, 쇼리뷰 l 보여주는 맛집, 치하루 Chiharu, 컬러템포 ColorTempo

📍 제주 서귀포시 안덕면 난드르로 49-17  — 영상 3개 / 채널 3곳
   가보자곰, 공간기록자, 올튜브

📍 제주 서귀포시 성산읍 일출로 222  — 영상 3개 / 채널 3곳
   제제엠의 여기저기, 제주도에미친녀자 제미녀, 치하루 Chiharu

📍 제주 제주시 애월읍 하귀로 62  — 영상 3개 / 채널 3곳
   럭키커플트립 Lucky Couple Trip, 치하루 Chiharu, 컬러템포 ColorTempo

📍 제주 제주시 한림읍 옹포7길 25-3  — 영상 3개 / 채널 3곳
   JINI지니의 기록지침, 샤슐랭 가이드, 컬러템포 ColorTempo

📍 제주 서귀포시 안덕면 사계남로216번길 24-61  — 영상 3개 / 채널 3곳
   공간기록자, 럭키커플트립 Lucky Couple Trip, 져니로그 Journey Log

📍 제주시 구좌읍 해맞이해안로 2196  — 영상 2개 / 채널 2곳
   루씨앤정, 여행작가 봄비

📍 제주시 구좌읍 김녕로1길 51-3  — 영상 3개 / 채널 2곳
   Drone_Korea, 여행작가 봄비

📍 제주 제주시 구좌읍 행원로1길 32-3  — 영상 2개 / 채널 2곳
   제주사는 제주여행자, 푸글

📍 제주시 애월읍 애월북서길 60  — 영상 2개 / 채널 2곳
   살란다, 섬 좋아하는 취향


In [3]:
# 노트북 셀 — 카카오 로컬 첫 실측
import requests
from config import KAKAO_KEY

H = {"Authorization": f"KakaoAK {KAKAO_KEY}"}

# 케이스 1: 장소명 검색 (경로 2 — Pass 2 겸용)
r1 = requests.get(
    "https://dapi.kakao.com/v2/local/search/keyword.json",
    headers=H, params={"query": "제주 오조포구", "size": 3},
).json()
print("=== 오조포구 검색 ===")
for d in r1.get("documents", []):
    print(f"  {d['place_name']} | {d['category_name']}")
    print(f"    {d['road_address_name'] or d['address_name']}")
    print(f"    좌표 ({d['y']}, {d['x']}) | place_id {d['id']}")

# 케이스 2: 주소 → 좌표 (경로 1 — 다수결 1위로 테스트)
r2 = requests.get(
    "https://dapi.kakao.com/v2/local/search/address.json",
    headers=H, params={"query": "제주 제주시 한림읍 한림서길 18"},
).json()
print("\n=== 한림서길 18 변환 ===")
for d in r2.get("documents", []):
    print(f"  ({d['y']}, {d['x']}) | {d['address_name']}")

# 보너스: 한림서길 18이 대체 뭐 하는 곳인지 — 좌표 역검색
if r2.get("documents"):
    y, x = r2["documents"][0]["y"], r2["documents"][0]["x"]
    r3 = requests.get(
        "https://dapi.kakao.com/v2/local/search/keyword.json",
        headers=H, params={"query": "카페", "y": y, "x": x, "radius": 50, "size": 5},
    ).json()
    print("\n=== 그 자리 반경 50m ===")
    for d in r3.get("documents", []):
        print(f"  {d['place_name']} | {d['category_name']}")

=== 오조포구 검색 ===

=== 한림서길 18 변환 ===


In [4]:
import requests
from config import KAKAO_KEY

H = {"Authorization": f"KakaoAK {KAKAO_KEY}"}
r = requests.get(
    "https://dapi.kakao.com/v2/local/search/keyword.json",
    headers=H, params={"query": "제주 오조포구", "size": 3},
)
print(r.status_code)
print(r.json())

403
{'errorType': 'NotAuthorizedError', 'message': 'App(제주trip) disabled OPEN_MAP_AND_LOCAL service.'}


In [3]:
# ── 카페 보강 수집 (완결형: 시드 로드 → 수집 → 저장 → 코퍼스 필터) ──
import json, time, os, re
from datetime import datetime

CAFE_KEYWORDS = [
    # 유형/무드 (광역)
    "제주 오션뷰 카페", "제주 감성 카페", "제주 대형카페",
    "제주 베이커리 카페", "제주 브런치 카페", "제주 디저트 카페",
    "제주 애견동반 카페", "제주 조용한 카페", "제주 신상 카페",
    "제주 노을 카페",
    # 지역 확장 (카페 성지)
    "곽지 카페", "김녕 카페", "세화 카페", "조천 카페",
    "남원 카페", "위미 카페", "사계 카페", "대정 카페",
]

# 1) 기존 raw를 시드로 — 재적중 시 keywords만 자람 (호출 절약 + 다수결 증폭)
SEED = "data/raw/raw_20260707_1006.json"
with open(SEED, encoding="utf-8") as f:
    collected = {v["video_id"]: v for v in json.load(f)}
seed_n = len(collected)
rehit = 0
print(f"시드 로드: {seed_n}건")

# 2) 수집 루프
for kw in CAFE_KEYWORDS:
    try:
        resp = yt.search().list(
            part="snippet", q=kw, type="video",
            regionCode="KR", relevanceLanguage="ko",
            maxResults=50, order="relevance",
            publishedAfter="2023-07-01T00:00:00Z",
        ).execute()
        ids = [i["id"]["videoId"] for i in resp.get("items", [])]

        new_ids = []
        for vid in ids:
            if vid in collected:
                if kw not in collected[vid]["keywords"]:
                    collected[vid]["keywords"].append(kw)
                    rehit += 1
            else:
                new_ids.append(vid)

        for j in range(0, len(new_ids), 50):
            vr = yt.videos().list(
                part="snippet,statistics,contentDetails",
                id=",".join(new_ids[j:j+50]),
            ).execute()
            for v in vr.get("items", []):
                sn = v["snippet"]
                collected[v["id"]] = {
                    "video_id": v["id"],
                    "title": sn["title"],
                    "description": sn.get("description", ""),
                    "tags": sn.get("tags", []),
                    "channel_id": sn["channelId"],
                    "channel_title": sn["channelTitle"],
                    "published_at": sn["publishedAt"],
                    "view_count": int(v.get("statistics", {}).get("viewCount", 0)),
                    "duration": v.get("contentDetails", {}).get("duration", ""),
                    "keywords": [kw],
                }
        print(f"[{kw}] 검색 {len(ids)} / 신규 {len(new_ids)}")
        time.sleep(0.3)
    except Exception as e:
        print(f"[{kw}] 실패: {e} — 계속")

# 3) 저장 (raw 불변 원칙 — 새 버전 파일)
out = f"data/raw/raw_{datetime.now():%Y%m%d}_cafe.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(list(collected.values()), f, ensure_ascii=False, indent=2)

# 4) 카페 코퍼스 필터 — Pass 1 대상 확정
def is_cafe(v):
    kw_hit = any("카페" in k for k in v["keywords"])
    title_hit = bool(re.search(r"카페|cafe|커피|베이커리|브런치|디저트", v["title"], re.I))
    return kw_hit or title_hit

corpus = [v for v in collected.values() if is_cafe(v)]
cor_out = "data/processed/cafe_corpus.json"
os.makedirs("data/processed", exist_ok=True)
with open(cor_out, "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)

# 5) 요약
total = len(collected)
b_track = sum(1 for v in corpus if len(v["description"]) <= 100)
multi = sum(1 for v in corpus if len(v["keywords"]) >= 2)
print(f"\n전체 raw: {seed_n} → {total} (신규 {total - seed_n})")
print(f"기존 영상 카페 키워드 재적중: {rehit}건  ← 다수결 증폭")
print(f"카페 코퍼스: {len(corpus)}건 (B트랙 {b_track} / 복수 키워드 {multi})")
print(f"저장: {out}\n코퍼스: {cor_out}")

시드 로드: 1359건
[제주 오션뷰 카페] 검색 50 / 신규 27
[제주 감성 카페] 검색 50 / 신규 34
[제주 대형카페] 검색 50 / 신규 40
[제주 베이커리 카페] 검색 50 / 신규 32
[제주 브런치 카페] 검색 50 / 신규 37
[제주 디저트 카페] 검색 50 / 신규 34
[제주 애견동반 카페] 검색 50 / 신규 49
[제주 조용한 카페] 검색 50 / 신규 7
[제주 신상 카페] 검색 50 / 신규 23
[제주 노을 카페] 검색 50 / 신규 27
[곽지 카페] 검색 50 / 신규 47
[김녕 카페] 검색 50 / 신규 37
[세화 카페] 검색 50 / 신규 31
[조천 카페] 검색 50 / 신규 43
[남원 카페] 검색 50 / 신규 44
[위미 카페] 검색 50 / 신규 38
[사계 카페] 검색 50 / 신규 48
[대정 카페] 검색 50 / 신규 31

전체 raw: 1359 → 1988 (신규 629)
기존 영상 카페 키워드 재적중: 271건  ← 다수결 증폭
카페 코퍼스: 1169건 (B트랙 628 / 복수 키워드 233)
저장: data/raw/raw_20260707_cafe.json
코퍼스: data/processed/cafe_corpus.json


In [4]:
import json, random

with open("data/processed/cafe_corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"코퍼스: {len(corpus)}건")
for v in random.sample(corpus, 10):
    print(f"  [{','.join(v['keywords'][:2])}] {v['title'][:45]}")

코퍼스: 1169건
  [제주 감성 카페] 심야살롱 #제주 감성카페 #제주 오마이살롱 #제주감성숙소 #제주 오마이코티지 #서
  [김녕 카페] [vlog] 나혼자 제주 동쪽 여행🏡 | 걷다가 만난 뜻밖의 행복들❤️느좋카페,제
  [제주 브런치 카페] 제주여행만 20번🏝️ 혼밥, 혼카페 맛집 🍽️ ㅣ무조건 사진 건지는 카페들📸ㅣ비오
  [곽지 카페] 제주카페 추천👍 곽지해수욕장 바다뷰 카페#shorts
  [성산 카페] 🏝️색감이 너무 예쁜 제주 성산읍 대형 베이커리 카페 [서귀피안]
  [월정리 카페] (급매)월정 해안도로+엄청 넓은 주차장! 구좌읍 월정리 2층 상가건물 매매(2층/
  [중문 카페] 제주 카페(중문) 추천 | 바다바라카페 #shorts
  [우도 카페,우도 맛집] 제주 커플여행🍊 우도·맛집·카페 3박4일 브이로그 | 스타일 칭찬받은 여름휴가🏝️
  [위미 카페] 아빠가 뛰는게 너무 슬퍼요ㅣ위미카페
  [제주 오션뷰 카페] 애월의 오션뷰. 개인적으로 제주 오션뷰 카페 1둥이었습니다. #제주도 #애월 #바


In [11]:
# ── Pass 1 카페 특화판: 완결형 (관통 → 전량 겸용) ──
import json, time, os
from openai import OpenAI, RateLimitError
from config import OPENAI_KEY

client = OpenAI(api_key=OPENAI_KEY)

REGIONS = "애월, 곽지, 한림, 협재, 함덕, 월정리, 세화, 김녕, 성산, 표선, 남원, 위미, 중문, 사계, 대정, 안덕, 우도, 구좌, 조천, 제주시내, 서귀포시내, 기타"

CAFE_TAGS = "오션뷰, 산방산뷰, 숲뷰, 노을, 감성, 조용함, 대형, 베이커리, 브런치, 디저트, 애견동반, 노키즈존, 키즈친화, 통창, 야외석, 루프탑, 주차편함, 웨이팅, 신상, 로컬"

SYSTEM = f"""유튜브 영상 정보에서 제주도 카페를 추출해 json으로만 답해.

## 규칙
- 카페만 추출 (베이커리·브런치·디저트 카페 포함). 일반 음식점·술집·숙소는 제외
- 제주도가 아닌 곳 제외
- region 판단 우선순위: ① 본문 주소 ② 본문 문맥 ③ 수집 키워드 힌트.
  해시태그의 지역명은 근거로 쓰지 마 (노출용 스팸 많음)
- 단, 해시태그의 고유 장소명은 카페 후보로 채택
- region: {REGIONS} 중 하나
- tags: 다음 사전에서만 선택, 본문에 근거 있는 것만 0~5개: {CAFE_TAGS}
- summary: 검색될 자연어 1~2문장. 다음 슬롯 중 본문에 실제로 있는 것만:
  뷰, 분위기, 시그니처 메뉴와 가격대, 웨이팅/혼잡도, 주차, 좌석 특성, 영업 특이사항.
  지어내지 마. 정보가 이름뿐이면 summary는 빈 문자열 ""
- address: 본문에 명시된 주소만 그대로, 없으면 null
- info_richness: 주소나 슬롯 2개 이상="high", 슬롯 1개="mid", 이름뿐="low"
- 입력이 제목·태그뿐일 수 있음 (설명란 없는 숏츠) — 카페명 특정에 집중
- 카페가 없으면 {{"spots": []}}

## 출력 (json만)
{{"spots": [{{"spot_name": "", "region": "", "tags": [], "summary": "",
"address": null, "info_richness": ""}}]}}"""

def build_input(v):
    kw = ", ".join(v.get("keywords", []))
    tags = " ".join(v.get("tags", [])[:15])
    if len(v["description"]) > 100:   # A트랙
        return f"[수집 키워드: {kw}]\n제목: {v['title']}\n설명: {v['description'][:1500]}\n태그: {tags}"
    return f"[수집 키워드: {kw}]\n제목: {v['title']}\n태그: {tags}\n(설명란 없음 — 카페명 특정)"

def extract(v, max_retry=5):
    for i in range(max_retry):
        try:
            resp = client.chat.completions.create(
                model="gpt-5-mini",
                response_format={"type": "json_object"},
                max_completion_tokens=4000,
                reasoning_effort="minimal",
                messages=[{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": build_input(v)}],
            )
            content = resp.choices[0].message.content or ""
            if not content.strip():
                print(f"  ⚠ 빈 응답 [{v['video_id']}] finish={resp.choices[0].finish_reason}")
                return []
            spots = json.loads(content).get("spots", [])
            for s in spots:   # 식별자는 코드가 운반
                s["video_id"] = v["video_id"]
                s["view_count"] = v["view_count"]
                s["duration"] = v["duration"]
                s["source_keywords"] = v.get("keywords", [])
            return spots
        except RateLimitError:
            time.sleep(2 ** i)
        except (json.JSONDecodeError, KeyError) as e:
            print(f"  ⚠ 파싱 실패 [{v['video_id']}]: {e}")
            return []
    print(f"  ⚠ 재시도 초과 [{v['video_id']}]")
    return []

# ── 실행 ──
LIMIT = None   # 관통 20 → 통과 확인 후 None으로 바꿔 전량

with open("data/processed/cafe_corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

targets = corpus[:LIMIT] if LIMIT else corpus
results = []
out_path = "data/processed/pass1_cafe.json"

for i, v in enumerate(targets):
    spots = extract(v)
    results.extend(spots)
    track = "A" if len(v["description"]) > 100 else "B"
    print(f"[{i+1}/{len(targets)}][{track}] {v['title'][:32]} → {len(spots)}개")
    if (i + 1) % 50 == 0:   # 중간 저장
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# ── 요약 ──
empty_sum = sum(1 for s in results if not s["summary"])
rich = {"high": 0, "mid": 0, "low": 0}
for s in results:
    rich[s.get("info_richness", "low")] = rich.get(s.get("info_richness", "low"), 0) + 1
print(f"\n카페 스팟 {len(results)}개 저장 → {out_path}")
print(f"info_richness 분포: {rich}")
print(f"summary 빈 문자열(무정보): {empty_sum}개  ← low와 대체로 일치해야 정상")

[1/1169][A] 제주여행 가면 무조건 여기 중문관광단지🌴 주변 볼거리 놀거 → 0개
[2/1169][B] 6월 여행 떠나기 좋은 제주 서귀포 가볼만한곳 BEST4  → 0개
[3/1169][A] 이대로 따라해도 성공함! 2박 3일 제주도 서쪽 여행 코스 → 0개
[4/1169][A] 제주 여행 ☘️ | 소품샵, 맛집, 카페 다 성공 ღ 해버 → 4개
[5/1169][B] 이 맛에 제주 사는거 아입니까 #제주도여행 #제주여행 #프 → 0개
[6/1169][B] 칭찬받는 제주 2박 3일 동쪽 여행코스 🍊 #제주여행 #제 → 0개
[7/1169][B] 비올때 가야하는 제주도 명소 BEST8☔️ #국내여행 #제 → 0개
[8/1169][A] 제주도에 오픈카타러 떠난 박미선 김정난 20년 우정여행 ( → 1개
[9/1169][A] vlog) 진짜 제주맛집을 찾아 제주여행브이로그 | 유럽같 → 0개
[10/1169][B] 이효리가 추천하는 제주 여행 코스 → 0개
[11/1169][B] 제주도맛집 필수코스 top4 → 0개
[12/1169][B] 2박 2일 제주여행 코스 🏡🤍  #쇼츠 #데일리룩 #oot → 0개
[13/1169][A] 5 Days in Jeju - Aesthetic Trave → 12개
[14/1169][A] 흔한 여행지❌ 제주 동쪽 희귀 스팟 여행 코스 가볼만한곳  → 0개
[15/1169][A] 제주여행 vlogㅣ이렇게 여행 와서 잘 먹고 힐링하려고 열 → 0개
[16/1169][A] 2026ver. 제주 3박 4일 가성비 여행 코스 총정리🏝 → 0개
[17/1169][A] 천만원쓰고 찾아낸 제주시 맛집 Best → 0개
[18/1169][B] 제주도민 추천 찐 맛집 ※메모 → 0개
[19/1169][B] 강력추천! 제주도에서 가장 인기가 많은 맛집 소개해 드릴게 → 0개
[20/1169][B] 3-4년간 모은 제주맛집&카페.zip(자세한 내용은 고정댓 → 0개
[21/1169][B] 제주 갈 때마다 한가득 사오는 

KeyboardInterrupt: 

In [7]:
# low/0개인데 원문은 풍부한 케이스 — 과잉 보수 검증
for v in targets:
    got = [s for s in results if s["video_id"] == v["video_id"]]
    if len(v["description"]) > 300 and (not got or all(s["info_richness"] == "low" for s in got)):
        print("=" * 60)
        print(f"제목: {v['title']}")
        print(f"설명(400자): {v['description'][:400]}")
        print(f"추출: {json.dumps(got, ensure_ascii=False, indent=1)[:300]}")

제목: 제주여행 가면 무조건 여기 중문관광단지🌴 주변 볼거리 놀거리 싹다 모음 (약천사, 중문색달해변, 주상절리, 국제평화센터 등)
설명(400자): 제주도 중문 가볼만한곳 싹다 모음집👇

1️⃣ #대포주상절리
- 입장료: 어른 2,000원/ 청소년, 어린이(7-24) 1,000원
- 운영시간: 오전 9시 ~ 오후 5시 30분
- #주상절리 제트보트 25,000원

2️⃣ #중문색달해변 - 제주 서핑의 성지
3️⃣ #천제연폭포 #선암교 전망대(한라산전망대, 별내린전망대)
4️⃣ #베릿내오름 : 중문관광단지 근처 숨은 오름!
- 주차: 서귀포시 중문관광로 181

5️⃣ #국제평화센터
- 비 오는 날 아이와 가기 좋은 전시관
- 입장료: 성인 1,500원, 어린이 1,000원
- 기획전시실: 평화 전시 공모전 오기영 선정작가전 (2024.09.09-10.23)
- 기획전시실 외에도 3개의 전시실과 작은 도서관, 야외 전시실 등이 있습니다. 볼거리가 쏠쏠.

추출: []
제목: 이대로 따라해도 성공함! 2박 3일 제주도 서쪽 여행 코스 가볼만한곳 베스트 (제주 서부 여행)
설명(400자): 제주 서쪽 코스 알려드릴게요. 👇
1️⃣ 무지개해안도로가 있는 도두동
2️⃣ 10분 내로 오를 수 있는 도두봉
3️⃣ 목마등대가 있는 이호테우해변
4️⃣ 소금빌레가 있는 구엄리돌염전
5️⃣ 그네 맛집인 수산봉
6️⃣ 걷기 좋은 한담해안산책로와
7️⃣ 아름다운 일몰이 펼쳐지는 곽지과물해변과 귀덕포구
8️⃣ 힐링카약파크인 비체올린
9️⃣ 마지막 수월봉지질트레일까지 

2박 3일 알찬 제주 서쪽 여행 코스
#여행작가봄비_제주
@iamhappyy

👉 #제주서쪽 #제주서쪽여행 #제주서부 #제주서부여행 #제주여행 #제주가볼만한곳 #제주여행코스 #제주핫플 #여행코스
#도두봉 #이호테우해변 #이호테우 #수산봉 #구엄리돌염전 #한담해안산책로 #곽지과물해변 #귀덕포구 #비체올린 #수월봉 #jeju #jejutravel #j
추출: []
제목: 제주 여행 ☘️ | 소품샵, 맛집, 카페 다 성공 

In [8]:
# 재관통 표본: 카페 키워드로 수집된 진짜 카페 영상에서
cafe_kw_hits = [v for v in corpus
                if any(k in CAFE_KEYWORDS for k in v.get("keywords", []))]
print(f"카페 키워드 수집분: {len(cafe_kw_hits)}건")

import random
random.seed(42)
targets = random.sample(cafe_kw_hits, min(20, len(cafe_kw_hits)))
# 이후 extract 루프 동일

카페 키워드 수집분: 752건


In [9]:
# ── 재관통: 카페 키워드 수집분에서 무작위 20건 ──
import random

cafe_kw_hits = [v for v in corpus
                if any(k in CAFE_KEYWORDS for k in v.get("keywords", []))]
print(f"카페 키워드 수집분: {len(cafe_kw_hits)}건")

random.seed(42)   # 재현 가능하게
targets = random.sample(cafe_kw_hits, 20)

results = []
out_path = "data/processed/pass1_cafe_pilot2.json"   # 1차 관통과 구분

for i, v in enumerate(targets):
    spots = extract(v)          # 기존 extract 함수 그대로 (SYSTEM만 보강판이어야 함)
    results.extend(spots)
    track = "A" if len(v["description"]) > 100 else "B"
    print(f"[{i+1}/20][{track}] {v['title'][:32]} → {len(spots)}개")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# 요약 + 판정 지표
rich = {"high": 0, "mid": 0, "low": 0}
bad_tags = []
for s in results:
    rich[s.get("info_richness", "low")] += 1
    for t in s.get("tags", []):
        if t not in CAFE_TAGS:
            bad_tags.append((s["spot_name"], t))
empty = sum(1 for s in results if not s["summary"])
gita = sum(1 for s in results if s["region"] == "기타")

print(f"\n스팟 {len(results)}개 | 분포 {rich} | 빈 summary {empty} | region기타 {gita}")
print(f"태그 사전 위반: {bad_tags or '없음'}")

카페 키워드 수집분: 752건
[1/20][A] 제주 일상 브이로그 | 서귀포 짤막한 2박3일 | 위미 바 → 7개
[2/20][B] 제주 유명 연예인 카페 베스트 5 → 0개
[3/20][A] 제주도 분위기 좋은 카페 6곳 추천 ☕️💛 #제주도여행 # → 0개
[4/20][B] 남양주 제주감성 브런치 맛집 카펜트리 → 0개
[5/20][A] 제주애월빵공장, 제주석양맛집베이커리카페, 제주디저트 #sh → 1개
[6/20][B] 제주시내 정원이 아름다운 대형베이커리카페 #제주핫플 #신상 → 1개
[7/20][B] 제주오션뷰미친젤예쁜카페ㅣ영상속카페는?ㅣ김녕,협재주변카페모음 → 1개
[8/20][B] 제주도민이 2년 동안 50번 이상 갔던 브런치 맛집..🩵  → 1개
[9/20][A] 제과명장 이흥용 칠암사계 세컨드 [금정사계] → 0개
[10/20][A] 보기 좋은 카페에서 먹기도 좋다. 조천가면 꼭 들르는 카페 → 1개
[11/20][A] [제주도 맛집] 함덕·조천·구좌 Best 맛집&카페 40곳 → 10개
[12/20][A] 남원 여행 일정. 맛집. 카페. 숙소 추천‼️ #남원여행 → 0개
[13/20][B] 제주 애월해안도로 핫플레이스 대형 베이커리 노을리 카페 → 1개
[14/20][A] 제주도 신상카페 top4☕️ #제주도카페 → 4개
[15/20][A] 제주에서 손가락에 꼽는 운치있는 카페  바로 여기우다여기  → 1개
[16/20][B] #성산카페 #제주동쪽 #제주동쪽여행 #플로이스트카페 #섭지 → 1개
[17/20][B] 제주도 대형카페 유리창청소 작업 브이로그🎬 → 0개
[18/20][B] 대왕 맘모스가 1,800원?! 모든게 다 수제인 빵집 #빵 → 0개
[19/20][A] 나혼자 제주도여행🫧• 실패없는 세화 카페, 맛집 여행코스  → 4개
[20/20][A] 한적한 소도시 감성, #남원 1박2일 코스 추천❤️ → 0개

스팟 33개 | 분포 {'high': 4, 'mid': 3, 'low': 26} | 빈 summary 25 | region기타

In [12]:
# ── Pass 1 전량 발사: 카페 코퍼스 752건+ ──
import json, time, os, random
from openai import OpenAI, RateLimitError
from config import OPENAI_KEY

client = OpenAI(api_key=OPENAI_KEY)

REGIONS = "애월, 곽지, 한림, 협재, 함덕, 월정리, 세화, 김녕, 성산, 표선, 남원, 위미, 중문, 사계, 대정, 안덕, 우도, 구좌, 조천, 제주시내, 서귀포시내, 기타"

CAFE_TAGS = "오션뷰, 산방산뷰, 숲뷰, 노을, 감성, 조용함, 대형, 베이커리, 브런치, 디저트, 애견동반, 노키즈존, 키즈친화, 통창, 야외석, 루프탑, 주차편함, 웨이팅, 신상, 로컬, 라이브공연, 전통찻집"

SYSTEM = f"""유튜브 영상 정보에서 제주도 카페를 추출해 json으로만 답해.

## 규칙
- 카페만 추출 (베이커리·브런치·디저트 카페 포함). 일반 음식점·술집·숙소는 제외
- 제주도가 아닌 곳 제외
- region 판단 우선순위: ① 본문 주소 ② 본문 문맥 ③ 수집 키워드 힌트.
  해시태그의 지역명은 근거로 쓰지 마 (노출용 스팸 많음)
- 본문에 지역 근거가 전혀 없으면 수집 키워드의 지역명을 region으로 사용
- 단, 해시태그의 고유 장소명은 카페 후보로 채택
- region: {REGIONS} 중 하나
- tags: 다음 사전에 있는 단어만, 본문에 근거 있는 것만 0~5개. 사전 밖 단어 절대 금지:
  {CAFE_TAGS}
- summary: 검색될 자연어 1~2문장. 다음 슬롯 중 본문에 실제로 있는 것만:
  뷰, 분위기, 시그니처 메뉴와 가격대, 웨이팅/혼잡도, 주차, 좌석 특성, 영업 특이사항.
  지어내지 마. 정보가 이름뿐이면 summary는 빈 문자열 ""
- address: 본문에 명시된 주소만 그대로, 없으면 null
- info_richness: 주소나 슬롯 2개 이상="high", 슬롯 1개="mid", 이름뿐="low"
- 입력이 제목·태그뿐일 수 있음 (설명란 없는 숏츠) — 카페명 특정에 집중
- 카페가 없으면 {{"spots": []}}

## 출력 (json만)
{{"spots": [{{"spot_name": "", "region": "", "tags": [], "summary": "",
"address": null, "info_richness": ""}}]}}"""

def build_input(v):
    kw = ", ".join(v.get("keywords", []))
    tags = " ".join(v.get("tags", [])[:15])
    if len(v["description"]) > 100:
        return f"[수집 키워드: {kw}]\n제목: {v['title']}\n설명: {v['description'][:1500]}\n태그: {tags}"
    return f"[수집 키워드: {kw}]\n제목: {v['title']}\n태그: {tags}\n(설명란 없음 — 카페명 특정)"

def extract(v, max_retry=5):
    for i in range(max_retry):
        try:
            resp = client.chat.completions.create(
                model="gpt-5-mini",
                response_format={"type": "json_object"},
                max_completion_tokens=4000,
                reasoning_effort="minimal",
                messages=[{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": build_input(v)}],
            )
            content = resp.choices[0].message.content or ""
            if not content.strip():
                print(f"  ⚠ 빈 응답 [{v['video_id']}]")
                return []
            spots = json.loads(content).get("spots", [])
            for s in spots:
                s["video_id"] = v["video_id"]
                s["view_count"] = v["view_count"]
                s["duration"] = v["duration"]
                s["source_keywords"] = v.get("keywords", [])
            return spots
        except RateLimitError:
            time.sleep(2 ** i)
        except (json.JSONDecodeError, KeyError) as e:
            print(f"  ⚠ 파싱 실패 [{v['video_id']}]: {e}")
            return []
    print(f"  ⚠ 재시도 초과 [{v['video_id']}]")
    return []

# ── 실행: 전량 ──
with open("data/processed/cafe_corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

targets = corpus   # 전량
results = []
out_path = "data/processed/pass1_cafe_full.json"
t0 = time.time()

for i, v in enumerate(targets):
    results.extend(extract(v))
    if (i + 1) % 25 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(targets) - i - 1) / 60
        print(f"[{i+1}/{len(targets)}] 스팟 {len(results)}개 누적 | 남은 시간 ~{eta:.0f}분")
    if (i + 1) % 50 == 0:   # 중간 저장 — 죽어도 진행분 보존
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# ── 출생신고서 ──
from collections import Counter
rich = Counter(s.get("info_richness", "low") for s in results)
regions = Counter(s["region"] for s in results)
names = len(set(s["spot_name"] for s in results))
addr = sum(1 for s in results if s.get("address"))
print(f"\n{'='*50}")
print(f"총 스팟 언급: {len(results)}개 / 고유 이름: {names}개")
print(f"분포: {dict(rich)} | 주소 보유: {addr}개")
print(f"지역 상위: {regions.most_common(8)}")
print(f"지역 하위(구멍 후보): {regions.most_common()[-6:]}")
print(f"소요: {(time.time()-t0)/60:.0f}분")

[25/1169] 스팟 13개 누적 | 남은 시간 ~27분
[50/1169] 스팟 104개 누적 | 남은 시간 ~51분
[75/1169] 스팟 159개 누적 | 남은 시간 ~57분
[100/1169] 스팟 204개 누적 | 남은 시간 ~56분
[125/1169] 스팟 235개 누적 | 남은 시간 ~50분
[150/1169] 스팟 277개 누적 | 남은 시간 ~49분
[175/1169] 스팟 309개 누적 | 남은 시간 ~47분
[200/1169] 스팟 350개 누적 | 남은 시간 ~45분
[225/1169] 스팟 389개 누적 | 남은 시간 ~44분
[250/1169] 스팟 415개 누적 | 남은 시간 ~42분
[275/1169] 스팟 452개 누적 | 남은 시간 ~40분
[300/1169] 스팟 483개 누적 | 남은 시간 ~38분
[325/1169] 스팟 529개 누적 | 남은 시간 ~37분
[350/1169] 스팟 556개 누적 | 남은 시간 ~35분
[375/1169] 스팟 593개 누적 | 남은 시간 ~34분
[400/1169] 스팟 617개 누적 | 남은 시간 ~32분
[425/1169] 스팟 640개 누적 | 남은 시간 ~30분
[450/1169] 스팟 676개 누적 | 남은 시간 ~29분
[475/1169] 스팟 705개 누적 | 남은 시간 ~28분
[500/1169] 스팟 730개 누적 | 남은 시간 ~26분
[525/1169] 스팟 782개 누적 | 남은 시간 ~25분
[550/1169] 스팟 796개 누적 | 남은 시간 ~24분
[575/1169] 스팟 886개 누적 | 남은 시간 ~23분
[600/1169] 스팟 928개 누적 | 남은 시간 ~22분
[625/1169] 스팟 945개 누적 | 남은 시간 ~21분
[650/1169] 스팟 972개 누적 | 남은 시간 ~20분
[675/1169] 스팟 1019개 누적 | 남은 시간 ~19분
[700/1169] 스팟 1041개 누적 | 남은 시간 ~18분
[725/1169] 스팟 1058개 누적

In [13]:
# 검수용 CSV 내보내기 — 원본은 JSON 유지, 이건 사람 눈 전용 뷰
import json, csv

with open("data/processed/pass1_cafe_full.json", encoding="utf-8") as f:
    spots = json.load(f)

with open("data/processed/review_cafe.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(["spot_name", "region", "richness", "tags", "summary",
                "address", "video_id", "views", "keywords"])
    for s in sorted(spots, key=lambda x: (x["region"], x["spot_name"])):
        w.writerow([s["spot_name"], s["region"], s["info_richness"],
                    "|".join(s["tags"]), s["summary"], s.get("address") or "",
                    s["video_id"], s["view_count"],
                    "|".join(s.get("source_keywords", []))])

print(f"{len(spots)}행 → review_cafe.csv (엑셀에서 바로 열림)")

1504행 → review_cafe.csv (엑셀에서 바로 열림)


In [15]:
# ── 데이터 탐색: 병합 사전 답사 ──
import json, re
from collections import Counter, defaultdict

with open("data/processed/pass1_cafe_full.json", encoding="utf-8") as f:
    spots = json.load(f)
with open("data/processed/cafe_corpus.json", encoding="utf-8") as f:
    vid2ch = {v["video_id"]: v["channel_title"] for v in json.load(f)}

# 이름 정규화 (병합 키 미리보기)
def norm(name):
    n = re.sub(r"\s+", "", name.lower())
    n = re.sub(r"^(카페|cafe)|(카페|cafe)$", "", n)
    n = re.sub(r"(제주점|애월점|본점|\d호점)$", "", n)
    return n or name

# ① 차트 미리보기 — 정규화 이름 기준 그룹핑
groups = defaultdict(list)
for s in spots:
    groups[(norm(s["spot_name"]), s["region"])].append(s)

ranked = sorted(groups.items(),
                key=lambda x: len(set(vid2ch.get(s["video_id"], "?") for s in x[1])),
                reverse=True)
print("=== ① 차트인 상위 15 (채널 수 기준) ===")
for (name, region), lst in ranked[:15]:
    chs = set(vid2ch.get(s["video_id"], "?") for s in lst)
    vids = set(s["video_id"] for s in lst)
    best = max(lst, key=lambda s: {"high": 2, "mid": 1, "low": 0}[s["info_richness"]])
    print(f"  {lst[0]['spot_name']} ({region}) | 채널 {len(chs)} / 영상 {len(vids)} | {best['info_richness']}")

# ② 병합 규모 예측
multi = sum(1 for _, lst in groups.items() if len(lst) >= 2)
multi_ch = sum(1 for _, lst in groups.items()
               if len(set(vid2ch.get(s["video_id"], "?") for s in lst)) >= 2)
print(f"\n=== ② 병합 예측 ===")
print(f"언급 {len(spots)} → 그룹 {len(groups)}개")
print(f"2회 이상 언급 그룹: {multi} | 교차 채널(진짜 차트인): {multi_ch}")

# ③ 태그 분포 — UI 칩과 골든셋의 재료
tag_c = Counter(t for s in spots for t in s["tags"])
print(f"\n=== ③ 태그 분포 ===")
for t, n in tag_c.most_common():
    print(f"  {t}: {n}")

# ④ high 카드 품질 샘플 — 임베딩 원료 검수
import random
random.seed(7)
highs = [s for s in spots if s["info_richness"] == "high" and s["summary"]]
print(f"\n=== ④ high 카드 샘플 5 (총 {len(highs)}개) ===")
for s in random.sample(highs, 5):
    print(f"\n  ● {s['spot_name']} ({s['region']}) {s['tags']}")
    print(f"    {s['summary']}")

=== ① 차트인 상위 15 (채널 수 기준) ===
  해지개 (애월) | 채널 14 / 영상 16 | high
  노을리 (애월) | 채널 14 / 영상 16 | high
  해일리카페 (성산) | 채널 10 / 영상 10 | high
  카페 한라산 (세화) | 채널 8 / 영상 8 | high
  프릳츠 제주성산점 (성산) | 채널 7 / 영상 7 | high
  유지커피웍스 (제주시내) | 채널 7 / 영상 7 | mid
  카페델문도 (함덕) | 채널 7 / 영상 7 | high
  제주당 (애월) | 채널 7 / 영상 7 | high
  델문도 김녕점 (김녕) | 채널 7 / 영상 7 | mid
  빵귿 (제주시내) | 채널 6 / 영상 6 | high
  콩카페 (곽지) | 채널 6 / 영상 6 | high
  베릴 (협재) | 채널 5 / 영상 5 | mid
  애월빵공장 (애월) | 채널 5 / 영상 6 | high
  카페더라이트 (성산) | 채널 5 / 영상 5 | high
  카페 오른 (성산) | 채널 5 / 영상 5 | mid

=== ② 병합 예측 ===
언급 1504 → 그룹 1153개
2회 이상 언급 그룹: 173 | 교차 채널(진짜 차트인): 162

=== ③ 태그 분포 ===
  베이커리: 325
  오션뷰: 313
  감성: 288
  디저트: 231
  브런치: 126
  주차편함: 88
  대형: 75
  애견동반: 70
  야외석: 44
  노을: 29
  신상: 22
  통창: 20
  조용함: 19
  로컬: 17
  전통찻집: 17
  포토존: 13
  루프탑: 13
  키즈친화: 10
  숲뷰: 7
  뷰: 6
  웨이팅: 6
  커피: 5
  커피맛집: 4
  산방산뷰: 4
  라이브공연: 2
  스페셜티: 2
  노키즈존: 2
  북카페: 2
  레트로: 1
  카페: 1
  말차: 1
  제주도감성: 1
  아포가토: 1
  차: 1
  제주오션뷰카페: 1
  포토존 없음: 1
  핸드드립: 1
  포토